# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is a yes/no question with an observed label: `is_declining_label`, derived from `trend_direction`. Per the toolkit, that shape starts with Logistic Regression, then Random Forest. Logistic Regression gives a readable set of coefficients to sanity-check against the Week 4 signal audit (tier, volume, CTR gap should matter). Random Forest is added after as the stronger, non-linear check, since some of the audited signals (volume, CTR volatility) looked like threshold effects rather than straight lines. Both output probabilities, which is what the baseline's ranked queue needs for a fair precision@K comparison. `trend_direction` and `trend_pct` are excluded from features since the label is derived from them directly.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

os.chdir("./../../")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"rows: {df.shape[0]}, decline rate: {df['is_declining_label'].mean():.3f}")
print(df["is_declining_label"].value_counts())

rows: 30000, decline rate: 0.542
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by `client_id`, using `GroupShuffleSplit`, 80/20. `content_id` and `client_id` are pseudonyms used only for grouping, never features. A random row split would let pages from the same client sit in both train and test, so the model could learn client-specific quirks instead of general signal. This mirrors the client-holdout split used in Week 2's leakage demo. No time-based split is needed since the label is a 30-day trend snapshot rather than a forward-looking window, and the data contract already excludes `trend_pct`/`trend_direction` from features.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

work = df.copy()
work["tier_avg_ctr"] = work.groupby("position_tier")["ctr"].transform("mean")
work["ctr_gap"] = work["tier_avg_ctr"] - work["ctr"]
good_tier = work["position_tier"].isin(["top_3", "page_1", "striking"]).astype(int)
enough_volume = (work["impressions_90d"] >= 100).astype(int)
positive_gap = (work["ctr_gap"] > 0).astype(int)
work["baseline_score"] = good_tier * enough_volume * positive_gap * work["ctr_gap"] * work["impressions_90d"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(work, groups=work["client_id"]))
train, test = work.iloc[train_idx].copy(), work.iloc[test_idx].copy()

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"train rows: {train.shape[0]}, test rows: {test.shape[0]}")
print(f"train decline rate: {train['is_declining_label'].mean():.3f}, test: {test['is_declining_label'].mean():.3f}")
print(f"client overlap between train and test: {len(overlap)}")

train rows: 23837, test rows: 6163
train decline rate: 0.550, test: 0.511
client overlap between train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same features excluded per the data contract, same metric as Week 4: precision@K. The baseline score is recomputed here from the same rule (tier + volume + CTR gap) so it lands on the identical held-out test rows as the models, not the Week 4 CSV, which mixed train and test.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position",
            "ctr", "word_count", "engagement_rate", "scroll_rate", "search_volume", "competition"]

Xtr = train[features].replace([np.inf, -np.inf], np.nan).fillna(0)
Xte = test[features].replace([np.inf, -np.inf], np.nan).fillna(0)
ytr, yte = train["is_declining_label"].values, test["is_declining_label"].values

scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(Xtr_scaled, ytr)
test["logreg_score"] = logreg.predict_proba(Xte_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
test["rf_score"] = rf.predict_proba(Xte)[:, 1]

base_score = test["baseline_score"].values
lr_score = test["logreg_score"].values
rf_score = test["rf_score"].values

rows = []
for k in (20, 50, 100):
    rows.append({
        "k": k,
        "baseline": round(precision_at_k(base_score, yte, k), 3),
        "logreg": round(precision_at_k(lr_score, yte, k), 3),
        "random_forest": round(precision_at_k(rf_score, yte, k), 3),
    })
comparison = pd.DataFrame(rows)
comparison["base_rate"] = round(yte.mean(), 3)
print(comparison.to_string(index=False))

  k  baseline  logreg  random_forest  base_rate
 20      0.50    0.70           0.50      0.511
 50      0.54    0.58           0.56      0.511
100      0.48    0.55           0.61      0.511


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random forest leans most on `impressions_90d` and `content_age_days`, then `ctr`, matching the Week 4 signal audit where volume and tier drove the strongest verdicts. Both are plausible: high-visibility, older pages are the ones with enough history to show a real trend either way, so the model is not leaning on anything suspiciously perfect, no leakage flag here.

In the top 20 logreg picks, several wrong cases have `trend_direction` of `new`, `flat`, or `up`, not `down`. Pages with `new` trend (little prior impression history) score high because they resemble low-position, low-CTR pages the model associates with decline, but they are actually pages with too little history to have a real trend yet. This is the model's main blind spot: it cannot separate "genuinely declining" from "too new/thin to tell," since neither the label nor the features contain a direct signal for data recency.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

imp = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, n_jobs=-1)
importances = sorted(zip(features, imp.importances_mean), key=lambda x: -x[1])
print("permutation importance, random forest")
for name, val in importances:
    print(f"{name}: {val:.4f}")

top20 = test.sort_values("logreg_score", ascending=False).head(20)
wrong = top20[top20["is_declining_label"] == 0]
print(f"\n{len(wrong)} of top 20 logreg picks were not actually declining")
print(wrong[["content_id", "impressions_90d", "content_age_days", "avg_position", "ctr", "trend_direction", "logreg_score"]].head(5))

permutation importance, random forest
impressions_90d: 0.0361
content_age_days: 0.0262
ctr: 0.0135
engagement_rate: 0.0068
avg_position: 0.0029
scroll_rate: 0.0027
competition: -0.0015
search_volume: -0.0024
word_count: -0.0082
days_since_last_update: -0.0089

6 of top 20 logreg picks were not actually declining
                 content_id  impressions_90d  content_age_days  avg_position  \
19965  content_9066e7c9a8ba              139               105          13.9   
22204  content_ef9bdd92b523                6               127           4.3   
27993  content_26d48a980581             1266               106           4.6   
12602  content_2f51ca262e66               15               105          18.9   
8521   content_0ea80678706d              122               106          31.2   

       ctr trend_direction  logreg_score  
19965  0.0             new      0.684888  
22204  0.0            flat      0.683826  
27993  0.0              up      0.682485  
12602  0.0              up      0

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.